In [4]:
%pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu132

Looking in indexes: https://download.pytorch.org/whl/cu132
  Using cached torchvision-0.27.0%2Bcu132-cp310-cp310-win_amd64.whl.metadata (5.6 kB)
   ---------------------------------------- 0.0/8.5 MB ? eta -:--:--
   -------------- ------------------------- 3.1/8.5 MB 18.4 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.5 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 8.5/8.5 MB 17.5 MB/s  0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.27.0
    Uninstalling torchvision-0.27.0:
      Successfully uninstalled torchvision-0.27.0
Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install -U torch torchvision

Note: you may need to restart the kernel to use updated packages.


ERROR: You must give at least one requirement to install (maybe you meant "pip install https://download.pytorch.org/whl/torch_stable.html"?)


In [7]:
%pip install -r requirements.txt --upgrade --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


In [5]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import sys


In [6]:
print(sys.executable)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.backends.cudnn.is_available())

D:\python 3.10\python.exe
True
13.2
True


In [17]:

# YOLO - detekcja
model = YOLO("yolov8m.pt")
model.to("cuda")

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=

In [18]:
# Zmienne do Optical Flow
prev_gray = None #poprzednia klatka w skali szarości (Optical Flow)
prev_blur = None #poprzednia klatka w skali szarości (Optical Flow)
p0 = None  # punkt do ślledzenia (x,y - współrzędne)
trajectory = []  # historia ruchu - dlatego tak fajnie rysuje

In [19]:

# Video
cap = cv2.VideoCapture("data/Kamera_taktyczne_male_ruchy.mp4")

In [20]:
# Tworzymy filtr Kalmana:
# 4 zmienne stanu: x, y, vx, vy
# 2 pomiary: x, y

kalman = cv2.KalmanFilter(4, 2)

# Macierz przejścia stanu (jak zmienia się stan w czasie)
# x = x + vx
# y = y + vy
kalman.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=np.float32)

# Macierz pomiaru (co obserwujemy: tylko x i y)
kalman.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], dtype=np.float32)

# Szum procesu (jak bardzo model może się mylić)
kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2

# Szum pomiaru (jak bardzo YOLO może się mylić)
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1

# Początkowa niepewność estymacji
kalman.errorCovPost = np.eye(4, dtype=np.float32)

# Początkowy stan (x, y, vx, vy)
kalman.statePost = np.zeros((4, 1), dtype=np.float32)
kalman_initialized = False

In [21]:
paused = False  # flaga pauzy

try:
    while cap.isOpened():
        # --- Obsługa pauzy ---
        if paused:
            key = cv2.waitKey(1) & 0xFF
            if key == ord(' '):       # spacja – wznowienie
                paused = False
            elif key == ord('q'):     # 'q' – zakończ nawet w trakcie pauzy
                break
            continue  # pomiń resztę pętli, nie odczytuj nowej klatki

        ret, frame = cap.read()  # Wczytujemy klatkę
        if not ret:
            break

        # Konwersja do skali szarości (wymagane do Optical Flow)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

                # ---------- OKNO GĘSTEGO PRZEPŁYWU OPTYCZNEGO (kolorowa wizualizacja) ----------
        flow_vis = None
        if prev_gray is not None:
            # Oblicz gęsty przepływ optyczny metodą Farnebäcka
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                0.5, 3, 15, 3, 5, 1.2, 0
            )
            # Oblicz wielkość i kąt dla każdego wektora
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            # Mapowanie: kąt -> barwa (H), wielkość -> nasycenie (S)
            hsv = np.zeros((gray.shape[0], gray.shape[1], 3), dtype=np.uint8)
            hsv[..., 0] = ang * 180 / np.pi / 2          # Hue: 0-180
            hsv[..., 1] = 255                            # Pełne nasycenie
            hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # Value z wielkości
            flow_vis = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


        # Możesz zakomentować / usunąć linię:
        # cv2.imshow("Optical Flow - Gray", gray)

        # Rozmycie Gaussa – redukcja szumu
        blur = cv2.GaussianBlur(gray, (3, 3), 0)

        # Binarna reprezentacja ruchu (różnica klatek)
        if prev_blur is not None:
            diff = cv2.absdiff(blur, prev_blur)
            _, motion_mask = cv2.threshold(diff, 12, 255, cv2.THRESH_BINARY)
        else:
            motion_mask = np.zeros_like(blur)

        track_point = None # aktualny punkt śledzenia
        ball_crop = None # wycięty obszar wokół piłki

        # Predykcja Kalmana tylko raz na klatkę
        predicted_point = None
        if kalman_initialized:
            pred = kalman.predict()
            predicted_point = (int(pred[0][0]), int(pred[1][0]))

        # OPTICAL FLOW (Lucas-Kanade)
        # Działa nawet wtedy kiedy YOLO chwilowo zawiedzie
        if prev_gray is not None and p0 is not None:

            # Obliczamy przesunięcie punktu między klatkami
            p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, p0, None)

            # Jeśli punkt został poprawnie znaleziony
            if st is not None and st[0][0] == 1:
                x, y = int(p1[0][0][0]), int(p1[0][0][1])

                # Rysujemy punkt (niebieski)
                cv2.circle(frame, (x, y), 5, (255, 0, 0), -1)

                track_point = (x, y)
                predicted_point = (x, y)  # ROI może bazować też na Optical Flow
                # aktualizujemy punkt do następnej iteracji
                p0 = p1
            else:
                # zgubiliśmy punkt
                p0 = None

        # YOLO DETEKCJA na całej klatce
        results = model(frame, conf=0.18, classes=[32], imgsz=1280)

        candidates = []

        # Kandydaci z pełnej klatki
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            candidates.append((x1, y1, x2, y2, conf, "full"))

        # YOLO DETEKCJA na przybliżonym ROI wokół przewidywanej pozycji piłki
        if predicted_point is not None:
            px, py = predicted_point

            roi_margin = 120
            roi_scale = 3

            roi_x1 = max(0, px - roi_margin)
            roi_y1 = max(0, py - roi_margin)
            roi_x2 = min(frame.shape[1], px + roi_margin)
            roi_y2 = min(frame.shape[0], py + roi_margin)

            roi_frame = frame[roi_y1:roi_y2, roi_x1:roi_x2].copy()

            if roi_frame.size > 0:
                # Wersja powiększona dla YOLO
                roi_for_yolo = cv2.resize(
                    roi_frame,
                    None,
                    fx=roi_scale,
                    fy=roi_scale,
                    interpolation=cv2.INTER_CUBIC
                )

                roi_results = model(roi_for_yolo, conf=0.12, classes=[32], imgsz=640)

                for box in roi_results[0].boxes:
                    rx1, ry1, rx2, ry2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])

                    # Przeliczamy współrzędne z powiększonego ROI na pełną klatkę
                    x1 = roi_x1 + int(rx1 / roi_scale)
                    y1 = roi_y1 + int(ry1 / roi_scale)
                    x2 = roi_x1 + int(rx2 / roi_scale)
                    y2 = roi_y1 + int(ry2 / roi_scale)

                    candidates.append((x1, y1, x2, y2, conf, "roi"))

        best = None # najlepszy bounding box
        best_score = -1 # najlepszy wynik

        # Parametry filtrowania — dostosuj do swojego wideo
        MIN_BALL_SIZE = 5
        MAX_BALL_SIZE = 40
        MIN_ASPECT = 0.5
        MAX_ASPECT = 2.0
        MAX_DIST_FROM_TRACK = 150

        for x1, y1, x2, y2, conf, source in candidates:
            box_w = x2 - x1
            box_h = y2 - y1
            box_size = max(box_w, box_h)
            aspect = box_w / (box_h + 1e-6)

            # Filtrowanie po rozmiarze — buty są często większe lub mniejsze niż piłka
            if box_size < MIN_BALL_SIZE or box_size > MAX_BALL_SIZE:
                continue

            # Filtrowanie po proporcjach — piłka jest okrągła, buty wydłużone
            if aspect < MIN_ASPECT or aspect > MAX_ASPECT:
                continue

            # Środek bounding boxa
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            # Odległość od predykcji
            if predicted_point is not None:
                px, py = predicted_point
                dist = np.hypot(cx - px, cy - py)
            else:
                dist = 0.0

            # Jeśli już śledzimy, odrzucamy detekcje daleko od trajektorii
            if kalman_initialized and dist > MAX_DIST_FROM_TRACK:
                continue

            # DETEKCJA RUCHU (różnica klatek)
            motion_ratio = 0.0
            if prev_blur is not None:
                diff = cv2.absdiff(blur, prev_blur)
                _, diff = cv2.threshold(diff, 12, 255, cv2.THRESH_BINARY)

                roi = diff[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
                if roi.size > 0:
                    motion_ratio = np.count_nonzero(roi) / roi.size

            # FILTRACJA: ignorujemy obiekty bez ruchu (na starcie)
            if motion_ratio < 0.50:
                if not kalman_initialized:
                    continue
                elif dist > 40:
                    continue

            # Bonus dla detekcji z ROI
            roi_bonus = 0.25 if source == "roi" else 0.0

            # Kara za wydłużony kształt (buty)
            roundness_penalty = abs(1.0 - aspect) * 0.3

            # SCORING
            score = conf - 0.025 * dist + 2.0 * motion_ratio + roi_bonus - roundness_penalty

            if score > best_score:
                best_score = score
                best = (cx, cy, x1, y1, x2, y2, source)

        # AKTUALIZACJA WYBRANEGO OBIEKTU
        if best is not None:
            cx, cy, x1, y1, x2, y2, source = best

            # Rysujemy bbox
            color = (255, 0, 255) if source == "roi" else (0, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Rysujemy środek (czerwony)
            cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

            # Informacja, skąd pochodzi detekcja
            cv2.putText(
                frame,
                source,
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

            # --- NOWA LOGIKA: Dynamiczny margines dla podglądu piłki ---
            box_w = x2 - x1
            box_h = y2 - y1
            ball_size = max(box_w, box_h)
            crop_margin = max(20, int(ball_size * 1.5))  # 1.5x rozmiar piłki, minimum 20 px

            CROP_DISPLAY_SIZE = 300  # Stały rozmiar okna

            crop_x1 = max(0, cx - crop_margin)
            crop_y1 = max(0, cy - crop_margin)
            crop_x2 = min(frame.shape[1], cx + crop_margin)
            crop_y2 = min(frame.shape[0], cy + crop_margin)

            ball_crop = frame[crop_y1:crop_y2, crop_x1:crop_x2].copy()

            if ball_crop.size > 0:
                ball_crop = cv2.resize(
                    ball_crop,
                    (CROP_DISPLAY_SIZE, CROP_DISPLAY_SIZE),
                    interpolation=cv2.INTER_CUBIC
                )
            # ---------------------------------------------------------

            # Aktualizacja Kalmana
            measurement = np.array([[np.float32(cx)], [np.float32(cy)]])
            if not kalman_initialized:
                # Pierwsza inicjalizacja Kalmana
                kalman.statePre = np.array([[cx], [cy], [0], [0]], dtype=np.float32)
                kalman.statePost = np.array([[cx], [cy], [0], [0]], dtype=np.float32)
                kalman_initialized = True
            else:
                # Korekta na podstawie pomiaru
                kalman.correct(measurement)

            # USTAWIENIE PUNKTU DO OPTICAL FLOW
            p0 = np.array([[[np.float32(cx), np.float32(cy)]]])
            track_point = (cx, cy)

        # RYSOWANIE TRAJEKTORII
        if track_point is not None and (not trajectory or track_point != trajectory[-1]):
            trajectory.append(track_point)

            # Ograniczamy długość ścieżki
            if len(trajectory) > 120:
                trajectory = trajectory[-120:]

        # Rysujemy linię trajektorii
        for i in range(1, len(trajectory)):
            cv2.line(frame, trajectory[i - 1], trajectory[i], (255, 0, 0), 2)

        # Zapisujemy klatkę jako poprzednią
        prev_gray = gray.copy()
        prev_blur = blur.copy()

        cv2.imshow("YOLO + Optical Flow", frame)
        cv2.imshow("Optical Flow - Gray", motion_mask)

        # Wyświetl to okno (zastąpi lub dodaj do istniejących)
        if flow_vis is not None:
            cv2.imshow("Optical Flow - ruch (kolor)", flow_vis)

        if ball_crop is not None and ball_crop.size > 0:
            cv2.imshow("Ball Crop / ROI", ball_crop)

        # Sprawdzanie klawiszy – spacja włącza pauzę, 'q' kończy
        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            paused = True
        elif key == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)


0: 736x1280 1 sports ball, 36.3ms
Speed: 7.1ms preprocess, 36.3ms inference, 2.2ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 sports balls, 33.2ms
Speed: 6.2ms preprocess, 33.2ms inference, 2.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 33.4ms
Speed: 6.1ms preprocess, 33.4ms inference, 1.3ms postprocess per image at shape (1, 3, 736, 1280)

0: 640x640 2 sports balls, 17.4ms
Speed: 3.9ms preprocess, 17.4ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 736x1280 1 sports ball, 33.6ms
Speed: 6.6ms preprocess, 33.6ms inference, 1.6ms postprocess per image at shape (1, 3, 736, 1280)

0: 640x640 1 sports ball, 16.8ms
Speed: 3.3ms preprocess, 16.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 736x1280 1 sports ball, 35.2ms
Speed: 6.1ms preprocess, 35.2ms inference, 1.7ms postprocess per image at shape (1, 3, 736, 1280)

0: 640x640 1 sports ball, 17.8ms
Speed: 3.3ms preprocess, 17.8ms inf